# Make All Tables for Main Manuscript

Kendra Wyant  
June 7, 2024

In [ ]:

suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(source("https://github.com/jjcurtin/lab_support/blob/main/format_path.R?raw=true"))
suppressPackageStartupMessages(library(tidyposterior))
library(kableExtra)



Attaching package: 'kableExtra'

The following object is masked from 'package:dplyr':

    group_rows

## Data and calculations

Table 1

In [ ]:
disposition <- read_csv(file.path(path_processed, "disposition.csv"), 
                        col_types = "ccDDcccccccccc")

screen <- read_csv(file.path(path_shared, "screen.csv"), 
                   col_types = cols()) |>
  filter(subid %in% subset(disposition, analysis == "yes")$subid) |> 
  mutate(across(dsm5_1:dsm5_11, ~ recode(., "No" = 0, "Yes" = 1))) |>  
  rowwise() |>  
  mutate(dsm5_total = sum(c(dsm5_1, dsm5_2, dsm5_3, dsm5_4, dsm5_5, dsm5_6, dsm5_7, 
                              dsm5_8, dsm5_9, dsm5_10, dsm5_11))) |>  
  ungroup()

lapses <- read_csv(file.path(path_shared, "lapses.csv"), col_types = cols()) |>
  filter(exclude == FALSE)

# Calcs to make df for table 1 (demographics and clinical characteristics)
n_total <- 151

dem_age <- screen |>
  summarise(mean = as.character(round(mean(dem_1, na.rm = TRUE), 1)),
            SD = as.character(round(sd(dem_1, na.rm = TRUE), 1)),
            min = as.character(min(dem_1, na.rm = TRUE)),
            max = as.character(max(dem_1, na.rm = TRUE))) |>
  mutate(var = "Age",
         n = as.numeric(""),
         perc = as.numeric("")) |>
  select(var, n, perc, everything()) 

dem_sex <-  screen |>
  select(var = dem_2) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |> 
  add_row(var = "Sex", .before = 1)

dem_race <- screen |>
  select(var = dem_3) |>
  mutate(var = fct_relevel(factor(var,
                         c("American Indian/Alaska Native", "Asian", "Black/African American",
                           "White/Caucasian", "Other/Multiracial")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Race", .before = 1)
  
  
dem_ethnicity <- screen |>
  select(var = dem_4) |>
  mutate(var = case_when(var == "No, I am not of Hispanic, Latino, or Spanish origin" ~ "No",
                         TRUE ~ "Yes"),
         var = fct_relevel(factor(var, c("Yes", "No")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Hispanic, Latino, or Spanish origin", .before = 1)

dem_education <- screen |>
  select(var = dem_5) |>
  mutate(var = fct_relevel(factor(var,
                         c("Less than high school or GED degree", "High school or GED",
                           "Some college", "2-Year degree", "College degree", "Advanced degree")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Education", .before = 1)

dem_employment <- screen |>
  select(var = dem_6, dem_6_1) |>
  mutate(var = case_when(dem_6_1 == "Full-time" ~ "Employed full-time",
                         dem_6_1 == "Part-time" ~ "Employed part-time",
                         TRUE ~ var)) |>
  mutate(var = fct_relevel(factor(var,
                         c("Employed full-time", "Employed part-time", "Full-time student",
                           "Homemaker", "Disabled", "Retired", "Unemployed",
                           "Temporarily laid off, sick leave, or maternity leave",
                           "Other, not otherwise specified")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |>
  add_row(var = "Employment", .before = 1)

dem_income <- screen |>
  summarise(mean = format(round(mean(dem_7, na.rm = TRUE), 0), big.mark = ","),
            SD = format(round(sd(dem_7, na.rm = TRUE), 0), big.mark = ","),
            min =format(round(min(dem_7, na.rm = TRUE), 0), big.mark = ","),
            max = format(round(max(dem_7, na.rm = TRUE), 0), scientific = FALSE, big.mark = ",")) |>
  mutate(var = "Personal Income",
        n = as.numeric(""),
        perc = as.numeric(""),
        mean = str_c("$", as.character(mean)),
        SD = str_c("$", as.character(SD)),
        min = str_c("$", as.character(min)),
        max = as.character(max)) |>
  select(var, n, perc, everything())

dem_marital <- screen |>
  select(var = dem_8) |>
  mutate(var = case_when(var == "Never Married" ~ "Never married",
                         TRUE ~ var)) |>
  mutate(var = fct_relevel(factor(var,
                         c("Never married", "Married", "Divorced", "Separated",
                           "Widowed")))) |>
  group_by(var) |>
  summarise(n = n()) |>
  mutate(perc = (n / sum(n)) * 100) |> 
  add_row(var = "Marital Status", .before = 1)

dem_aud <- screen |>
  summarise(mean = as.character(round(mean(dsm5_total, na.rm = TRUE), 1)),
            SD = as.character(round(sd(dsm5_total, na.rm = TRUE), 1)),
            min = as.character(min(dsm5_total, na.rm = TRUE)),
            max = as.character(max(dsm5_total, na.rm = TRUE))) |>
  mutate(var = "DSM-5 AUD Symptom Count",
         n = as.numeric(""),
         perc = as.numeric("")) |>
  select(var, n, perc, everything()) 

lapses_per_subid <- screen |>
  select(subid) |>
  left_join(lapses |>
  janitor::tabyl(subid) |>
  select(-percent), by = "subid") |>
  mutate(n = if_else(is.na(n), 0, n),
         lapse = if_else(n > 0, "yes", "no"))

lapse_info <- lapses_per_subid |>
  group_by(lapse) |>
  rename(var = lapse) |>
  mutate(var = factor(var, levels = c("yes", "no"), labels = c("Yes", "No"))) |>
  summarise(n = n()) |>
  mutate(perc = (n / n_total) * 100,
         mean = NA_character_,
         SD = NA_character_,
         min = NA_character_,
         max = NA_character_) |>
  full_join(lapses_per_subid |>
  summarise(mean = as.character(round(mean(n), 1)),
            SD = as.character(round(sd(n), 1)),
            min = as.character(round(min(n), 1)),
            max = as.character(round(max(n), 1))) |>
  mutate(var = "Number of reported lapses"),
  by = c("var", "mean", "SD", "min", "max")) |> 
  add_row(var = "Reported 1 or More Lapse During Study Period", .before = 1)

table_dem <- dem_age |> 
  bind_rows(dem_sex) |> 
  bind_rows(dem_race) |> 
  bind_rows(dem_ethnicity) |> 
  bind_rows(dem_education) |> 
  bind_rows(dem_employment) |> 
  bind_rows(dem_income) |> 
  bind_rows(dem_marital) |> 
  bind_rows(dem_aud) |> 
  bind_rows(lapse_info) |> 
  mutate(range = str_c(min, "-", max),
         perc = round(perc, 1)) |> 
  select(-c(min, max)) |> 
  rename(N = n,
         `%` = perc,
         M = mean, 
         Range = range)


Table 2

In [ ]:
pp_dem <- read_csv(file.path(path_models_lag, "pp_dem_0.csv"), col_types = cols()) |> 
  mutate(lag = 0) |> 
  bind_rows(read_csv(file.path(path_models_lag, "pp_dem_24.csv"), col_types = cols()) |> 
  mutate(lag = 24)) |> 
  bind_rows(read_csv(file.path(path_models_lag, "pp_dem_72.csv"), col_types = cols()) |> 
  mutate(lag = 72)) |> 
  bind_rows(read_csv(file.path(path_models_lag, "pp_dem_168.csv"), col_types = cols()) |> 
  mutate(lag = 168)) |> 
  bind_rows(read_csv(file.path(path_models_lag, "pp_dem_336.csv"), col_types = cols()) |> 
  mutate(lag = 336))

# pp_dem_contrast <- read_csv(file.path(path_models_lag, "pp_dem_contrast_0.csv"), col_types = cols()) |> 
#   mutate(lag = 0) |> 
#   bind_rows(read_csv(file.path(path_models_lag, "pp_dem_contrast_24.csv"), col_types = cols()) |> 
#   mutate(lag = 24)) |> 
#   bind_rows(read_csv(file.path(path_models_lag, "pp_dem_contrast_72.csv"), col_types = cols()) |> 
#   mutate(lag = 72)) |> 
#   bind_rows(read_csv(file.path(path_models_lag, "pp_dem_contrast_168.csv"), col_types = cols()) |> 
#   mutate(lag = 168)) |> 
#   bind_rows(read_csv(file.path(path_models_lag, "pp_dem_contrast_336.csv"), col_types = cols()) |> 
#   mutate(lag = 336))
# 
# pp_dem_contrast <- pp_dem_contrast |> 
#   group_by(contrast) |> 
#   summarise(across(probability:upper, median))


In [ ]:
pp_dem <- pp_dem |> 
  mutate(lag = factor(lag, levels = c(0, 24, 72, 168, 336), 
                        labels = c("0 lag", "24 lag", "72 lag", "168 lag", "336 lag" )),
         model = factor(model, levels = c("not white", "white",
                                        "female", "male",
                                        "below poverty", "above poverty",
                                        "older", "younger" ))) %>% 
  arrange(model, lag)

pp_dem_all <- pp_dem |> 
  filter(lag == "0 lag") |> 
  mutate(pp_lower = round(pp_lower, 3),
         pp_upper = round(pp_upper, 3),
         ci = str_c(pp_lower,"-",pp_upper)) |>
  select(-c(lag, pp_lower, pp_upper)) |> 
  bind_cols(pp_dem |> 
  filter(lag == "24 lag") |> 
  mutate(pp_lower = round(pp_lower, 3),
         pp_upper = round(pp_upper, 3),
         ci = str_c(pp_lower,"-",pp_upper)) |>
  select(-c(lag, pp_lower, pp_upper, model))) |> 
  bind_cols(pp_dem |> 
  filter(lag == "72 lag") |> 
  mutate(pp_lower = round(pp_lower, 3),
         pp_upper = round(pp_upper, 3),
         ci = str_c(pp_lower,"-",pp_upper)) |>
  select(-c(lag, pp_lower, pp_upper, model))) |> 
  bind_cols(pp_dem |> 
  filter(lag == "168 lag") |> 
  mutate(pp_lower = round(pp_lower, 3),
         pp_upper = round(pp_upper, 3),
         ci = str_c(pp_lower,"-",pp_upper)) |>
  select(-c(lag, pp_lower, pp_upper, model))) |> 
  bind_cols(pp_dem |> 
  filter(lag == "336 lag") |> 
  mutate(pp_lower = round(pp_lower, 3),
         pp_upper = round(pp_upper, 3),
         ci = str_c(pp_lower,"-",pp_upper)) |>
  select(-c(lag, pp_lower, pp_upper, model))) |> 
  add_row(model = "Race/Ethnicity", .before = 1) |> 
  add_row(model = "Sex", .before = 4) |> 
  add_row(model = "Income", .before = 7)  |> 
  add_row(model = "Age", .before = 10) |> 
  suppressMessages()


### Table 1: Demographic and Lapse Characteristics

In [ ]:

table_dem |> 
  knitr::kable()


  -------------------------------------------------------------------------------------------
  var                                            N     \% M          SD         Range
  ------------------------------------------ ----- ------ ---------- ---------- -------------
  Age                                                     41         11.9       21-72

  Sex                                                                           

  Female                                        74   49.0                       

  Male                                          77   51.0                       

  Race                                                                          

  American Indian/Alaska Native                  3    2.0                       

  Asian                                          2    1.3                       

  Black/African American                         8    5.3                       

  White/Caucasian                              131   86.8                       

  Other/Multiracial                              7    4.6                       

  Hispanic, Latino, or Spanish origin                                           

  Yes                                            4    2.6                       

  No                                           147   97.4                       

  Education                                                                     

  Less than high school or GED degree            1    0.7                       

  High school or GED                            14    9.3                       

  Some college                                  41   27.2                       

  2-Year degree                                 14    9.3                       

  College degree                                58   38.4                       

  Advanced degree                               23   15.2                       

  Employment                                                                    

  Employed full-time                            72   47.7                       

  Employed part-time                            26   17.2                       

  Full-time student                              7    4.6                       

  Homemaker                                      1    0.7                       

  Disabled                                       7    4.6                       

  Retired                                        8    5.3                       

  Unemployed                                    18   11.9                       

  Temporarily laid off, sick leave, or           3    2.0                       
  maternity leave                                                               

  Other, not otherwise specified                 9    6.0                       

  Personal Income                                         \$34,298   \$31,807   \$0-200,000

  Marital Status                                                                

  Never married                                 67   44.4                       

  Married                                       32   21.2                       

  Divorced                                      45   29.8                       

  Separated                                      5    3.3                       

  Widowed                                        2    1.3                       

  DSM-5 AUD Symptom Count                                 8.9        1.9        4-11

  Reported 1 or More Lapse During Study                                         
  Period                                                                        

  Yes                                           84   55.6                       

  No                                            67   44.4                       

  Number of reported lapses                               6.8        12         0-75
  -------------------------------------------------------------------------------------------


In [ ]:

table_dem |>
  head(-9) |>
  knitr::kable()


  -------------------------------------------------------------------------------------------
  var                                            N     \% M          SD         Range
  ------------------------------------------ ----- ------ ---------- ---------- -------------
  Age                                                     41         11.9       21-72

  Sex                                                                           

  Female                                        74   49.0                       

  Male                                          77   51.0                       

  Race                                                                          

  American Indian/Alaska Native                  3    2.0                       

  Asian                                          2    1.3                       

  Black/African American                         8    5.3                       

  White/Caucasian                              131   86.8                       

  Other/Multiracial                              7    4.6                       

  Hispanic, Latino, or Spanish origin                                           

  Yes                                            4    2.6                       

  No                                           147   97.4                       

  Education                                                                     

  Less than high school or GED degree            1    0.7                       

  High school or GED                            14    9.3                       

  Some college                                  41   27.2                       

  2-Year degree                                 14    9.3                       

  College degree                                58   38.4                       

  Advanced degree                               23   15.2                       

  Employment                                                                    

  Employed full-time                            72   47.7                       

  Employed part-time                            26   17.2                       

  Full-time student                              7    4.6                       

  Homemaker                                      1    0.7                       

  Disabled                                       7    4.6                       

  Retired                                        8    5.3                       

  Unemployed                                    18   11.9                       

  Temporarily laid off, sick leave, or           3    2.0                       
  maternity leave                                                               

  Other, not otherwise specified                 9    6.0                       

  Personal Income                                         \$34,298   \$31,807   \$0-200,000

  Marital Status                                                                

  Never married                                 67   44.4                       
  -------------------------------------------------------------------------------------------


In [ ]:

table_dem |>
  tail(9) |>
  knitr::kable()


  -------------------------------------------------------------------------------
  var                                                N     \% M     SD    Range
  ----------------------------------------------- ---- ------ ----- ----- -------
  Married                                           32   21.2             

  Divorced                                          45   29.8             

  Separated                                          5    3.3             

  Widowed                                            2    1.3             

  DSM-5 AUD Symptom Count                                     8.9   1.9   4-11

  Reported 1 or More Lapse During Study Period                            

  Yes                                               84   55.6             

  No                                                67   44.4             

  Number of reported lapses                                   6.8   12    0-75
  -------------------------------------------------------------------------------


### Table 2: Model performance by Demographic Group

In [ ]:

pp_dem_all |> 
  mutate(across(c(pp_median...2, pp_median...4, pp_median...6, pp_median...8, pp_median...10), 
                ~round(., 3)),
         across(where(is.numeric), as.character)) |> 
  add_row(model = "Group",
          pp_median...2 = "Median auROC",
          ci...3 = "Bayesian CI",
          pp_median...4 = "Median auROC",
          ci...5 = "Bayesian CI",
          pp_median...6 = "Median auROC",
          ci...7 = "Bayesian CI",
          pp_median...8 = "Median auROC",
          ci...9 = "Bayesian CI",
          pp_median...10 = "Median auROC",
          ci...11 = "Bayesian CI",
          .before = 1) |> 
  knitr::kable(col.names = c("", "0 Lag", "",
                    "24 Lag", "", 
                   "72 Lag", "",
                    "168 Lag", "",
                   "336 Lag", "")) 


  -----------------------------------------------------------------------------------------------------------------------------------
                   0 Lag                  24 Lag                 72 Lag                 168 Lag                336 Lag  
  ---------------- -------- ------------- -------- ------------- -------- ------------- -------- ------------- -------- -------------
  Group            Median   Bayesian CI   Median   Bayesian CI   Median   Bayesian CI   Median   Bayesian CI   Median   Bayesian CI
                   auROC                  auROC                  auROC                  auROC                  auROC    

  Race/Ethnicity                                                                                                        

  not white        0.736    0.635-0.815   0.696    0.608-0.773   0.645    0.469-0.79    0.635    0.467-0.784   0.589    0.367-0.783

  white            0.901    0.857-0.931   0.894    0.855-0.922   0.884    0.808-0.932   0.876    0.795-0.927   0.854    0.734-0.927

  Sex                                                                                                                   

  female           0.85     0.793-0.893   0.838    0.783-0.88    0.817    0.71-0.891    0.808    0.694-0.883   0.775    0.613-0.878

  male             0.921    0.886-0.945   0.917    0.887-0.94    0.909    0.848-0.946   0.917    0.86-0.953    0.9      0.804-0.951

  Income                                                                                                                

  below poverty    0.75     0.651-0.833   0.728    0.635-0.804   0.705    0.526-0.835   0.663    0.478-0.815   0.607    0.366-0.801

  above poverty    0.896    0.854-0.929   0.888    0.848-0.918   0.878    0.798-0.929   0.878    0.795-0.93    0.861    0.738-0.931

  Age                                                                                                                   

  older            0.89     0.838-0.928   0.84     0.779-0.886   0.902    0.823-0.949   0.886    0.792-0.941   0.951    0.884-0.981

  younger          0.903    0.862-0.932   0.897    0.86-0.925    0.886    0.809-0.933   0.881    0.801-0.931   0.859    0.735-0.932
  -----------------------------------------------------------------------------------------------------------------------------------
